# Your rating is noisier than your improvement

This is a note about measuring, not about strategy. There is no agent here and nothing to submit.

I spent several weeks trying to build better agents for Kaggriculture. I made very little progress that survived contact with the ladder. Eventually I worked out that the agents were not really the problem. The problem was that I could not tell whether any change I made was good or bad.

Here is the short version:

> **Submit the same agent twice and the two scores can land about 270 points apart. Rebuild the same policy and submit it twice and they can land about 400 points apart. So if your improvement is worth less than that, one submission cannot tell you whether it worked.**

If that is right, then a lot of the "+150 Elo" claims we all post, mine included, are measuring luck.


## 1. Two agents that disagreed with themselves

These are both my own submissions. In each row, both ratings come from the same policy. Not a variant. Not a retune. The same thing, submitted more than once.

| what I submitted | first score | episodes | second score | episodes | difference |
|---|---|---|---|---|---|
| the exact same file, same SHA hash | 2,230.8 | 109 | 2,499.0 | 124 | 268 |
| the same route, rebuilt the same way | 1,988.8 | 249 | 2,387.1 | 169 | 398 |

The first row is the cleanest. Identical bytes, identical hash, and both had played well past 91 episodes, which is roughly where ratings are supposed to have settled down. They still finished 268 points apart.

So I use two numbers rather than one. About 270 is the floor, measured on identical binaries. About 400 is what I have actually seen when the same policy is rebuilt and resubmitted.

Why does this happen? Three things stack up. You play a limited number of episodes, so there is ordinary sampling error. You are matched against whoever happens to be around, and their strength varies. And the whole field drifts over time as people submit new agents, so the same play is worth less next week than it was this week. None of that is a flaw in the ladder. It is just what a rating is.


In [ ]:
# The problem, in two numbers.
identical_binary_spread = 268   # the same file, same hash, submitted twice
rebuilt_policy_spread   = 398   # the same policy rebuilt, submitted twice
my_gap_to_top_10        = 355   # points I need to reach a prize spot

print("spread between two runs of an IDENTICAL binary:", identical_binary_spread)
print("spread when the same policy is rebuilt:       ", rebuilt_policy_spread)
print("distance I am trying to close:                ", my_gap_to_top_10)
print()
print("The larger figure alone exceeds the gap I am trying to close.")
print("Even the smaller one is three quarters of it.")
print("The thing I am measuring is about the size of the error in my ruler.")

## 2. What that rules out

Almost every improvement I have measured is smaller than that spread. Including all of mine.

| change I tested | what it was worth |
|---|---|
| smarter ordering of market sells | about 129 points |
| best possible sell order, using hindsight | about 25 points |
| structured edits to the action tape | 0 that carried over |

Every one of those sits far below 270. I spent two days carefully testing effects that a single submission cannot detect.

This does not mean small gains are useless. It means you cannot check them one submission at a time. One result is not evidence.


## 3. Your practice opponents are probably too easy

The obvious answer is to test offline instead. So I built a set of 258 real opponents from my own saved games and scored my agent against all of them.

| result | games |
|---|---|
| comfortable wins | 209 |
| narrow wins, under 8k | 37 |
| losses | 12 |

81 percent of those games are decided before they start. A set like that cannot rank your candidates, because nearly every game gives the same answer no matter what you change.

Here is what that cost me. I scored three of my agents against the 15 hardest opponents in that set, the ones where the games are actually close. They scored 0.733, 0.800 and 0.933. On the ladder, the one that scored **best** offline, 0.933, ended up rating **worst** of the three, at 1,920. The agent that scored 0.800 rated 2,495. That is a gap of about 575 points, and the practice set had them the other way round.

I am not saying the ladder is wrong. The ladder measures what it measures. What I am saying is that my offline ranking and the ladder ranking were not the same ranking, and I trusted the wrong one.

There is a reason my practice set was so easy. If you build it from games you played at your own rating, you are collecting opponents you already beat. Check how many of your games are actually close before you trust a number from them.


In [ ]:
# Worth running on your own practice set before you trust it.
comfortable, narrow, losses = 209, 37, 12
total = comfortable + narrow + losses
close = narrow + losses

print("opponents in the set:", total)
print("games that are actually close: %d (%.0f%%)" % (close, 100*close/total))
print("one game is worth %.4f of the win rate" % (1/total))
print()
print("To beat the noise, a change has to flip several of those %d games," % close)
print("not several of %d. That is a much harder bar than it looks." % total)

## 4. My offline testing got the order backwards, four times

I want to be specific about being wrong, because "my offline tests disagreed with the ladder" is easy to shrug off.

The 0.933 agent from section 3 is the clearest case. My practice set ranked it clearly ahead of the two I was already running. I submitted it on that basis. It settled at 1,920, about 575 points below the agent it was supposed to beat.

That was the fourth time my offline testing ordered candidates differently from the ladder.

I no longer use offline results to pick between agents that score close together. They are still good for one thing, which is telling a working agent apart from a broken one. That distinction is large and offline testing catches it reliably.


## 5. Two traps that cost me a morning

**What people publish is not always what they run.** I tested a public agent from an author ranked well above me. It lost 13 of its 15 games against opponents my own agent beats 12 out of 15. The likely reason is simply that people publish older or simpler versions than the ones they field, which is completely fair of them. But it means an author's rank tells you very little about the agent sitting in their notebook. Test everything before you submit it.

**Reposted agents come with new claims.** One agent I looked at turned out to be a policy I was already running. The action tape was identical. It had one line added. I checked whether that line did anything by setting it to an absurd value and running again, and all 15 games came back exactly the same, so the line never runs. Its title claimed a rating well above where I measure the same policy, though I have not verified that claim on the same episode window and it may not be comparable.

I am not naming anyone. I cannot rule out an honest mistake, and section 1 explains a difference that size on its own. That is really the point. When one agent can swing 270 to 400 points by itself, a bigger number next to someone's name is not proof their agent is better.


## 6. What I would suggest

1. Run the check in section 3 on your own practice set before you trust any number it gives you.
2. Never conclude anything from a single submission. If a change matters, it should survive more than one draw.
3. Keep a set of opponents you never tune against. Both of my opening searches looked better on the set I tuned on and came out worse on the ones I held back. Without those held back opponents I would have shipped both.
4. Test every agent you adopt before submitting it. My own failure case was an agent that loaded perfectly and then just returned the starting bank.
5. Treat a rating in a title as a claim, not a fact. Including mine.

Thank you to the people posting real, reproducible work with hashes and honest warnings about their own weak spots. That is how I learned how these agents are actually built, and several of those notebooks are more careful about their limits than I was about mine.

If you can prove any of this wrong, please do. I would rather be corrected than confident.
